# 1. Extracción de datos financieros — ISA, ISAGEN, EPM, CELSIA y Enel Colombia

**Proyecto:** Análisis de probabilidad de incumplimiento de emisores colombianos de energía.

## De PDF a XBRL

La primera versión de este proyecto extraía las cifras directamente de los PDF de resultados
financieros publicados por cada emisor, usando expresiones regulares para localizar cada partida
contable dentro del texto. Ese enfoque funcionó, pero exigió resolver un volumen considerable de
casos especiales -prosa que menciona un número y se confunde con un valor real, plantillas que
cambian de un trimestre a otro, unidades mezcladas dentro del mismo documento, columnas en orden
distinto según el año-, documentados en detalle en la versión anterior de este notebook.

Al descubrir que la Superintendencia Financiera de Colombia (RNVE) también publica estos mismos
reportes en formato **XBRL** (eXtensible Business Reporting Language), migramos a esa fuente para
la versión final del proyecto. La diferencia de fondo: en un PDF cada cifra es texto suelto que hay
que ubicar por contexto; en XBRL cada cifra es un **hecho etiquetado** con su concepto exacto de la
taxonomía IFRS (ej. `ifrs:Assets`) y su período exacto -sin ambigüedad de formato, sin importar en
qué trimestre ni qué plantilla haya usado la empresa ese año-.

**Resultado de la migración:** los trimestres marcados para revisión manual bajaron de 25 a 14 (de
131 a 146, porque XBRL amplió a Q3 2026 y llenó algunos huecos que el PDF no tenía), la completitud
de ISA, CELSIA e ISAGEN llegó al 100%, y no quedó ningún valor de ratio fuera de rango razonable en
el primer intento completo -contra las cerca de 20 rondas de corrección que hicieron falta con los
PDF-.

In [1]:
import sys, os
sys.path.append(os.path.abspath("../src"))

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import extract_xbrl as x
import build_dataset_xbrl
import compute_ratios_xbrl

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 1.1. Cómo funciona un archivo XBRL

Un archivo XBRL es un documento XML donde cada cifra ("hecho" o *fact*) está etiquetada con:

- **Un concepto** de la taxonomía (ej. `ifrs:Assets` para activo total, `ifrs:Revenue` para ingresos)
- **Un contexto**, que define el período (una fecha puntual para partidas de balance, o un rango
  fecha-inicio/fecha-fin para partidas del estado de resultados) y, cuando aplica, una **dimensión**
  (para desgloses por segmento de negocio, por país, por línea del activo, etc.)

La ventaja frente al PDF: no hay que "adivinar" qué número corresponde a qué partida por su posición
en la página -se busca el concepto exacto, en el período exacto, sin dimensión (para tomar el total
consolidado, no un desglose parcial)-.

In [2]:
# Estructura de los archivos descargados del RNVE
base = os.path.join("..", "data_xbrl")
conteo = {}
for empresa in sorted(os.listdir(base)):
    ruta_empresa = os.path.join(base, empresa)
    if not os.path.isdir(ruta_empresa):
        continue
    n = sum(len(files) for _, _, files in os.walk(ruta_empresa) if files)
    conteo[empresa] = n

pd.Series(conteo, name="Archivos XBRL").to_frame()

,Archivos XBRL
CELSIA,26
ENEL,30
EPM,30
ISA,30
ISAGEN,30


## 1.2. Cargar un XBRL con Arelle

Usamos [Arelle](https://arelle.org/), la librería estándar de código abierto para procesar XBRL en
Python, siguiendo la misma metodología que ya conocíamos de un ejercicio de referencia (cargar el
archivo con `Cntlr`, convertir sus *facts* en una tabla, y filtrar por concepto y fecha).

**Una diferencia respecto a ese ejercicio de referencia:** Arelle normalmente descarga el esquema
completo de la taxonomía (el DTS, *Discoverable Taxonomy Set*) desde los servidores de la
Superfinanciera y de XBRL International para poder resolver cada concepto con su tipo y definición
completos. En nuestro entorno de trabajo no hay salida a esos servidores, así que cargamos los
archivos con `skipDTS=True` -esto evita la descarga, y significa que `fact.concept` queda vacío,
pero `fact.qname` (el nombre del concepto tal como aparece en el propio XML) sigue funcionando
igual de bien para identificar cada hecho-.

In [3]:
facts_ejemplo = x.cargar_facts("../data_xbrl/ISA/2023/CIERRE 2023.xbrl")
print(f"Total de 'hechos' (facts) en este archivo: {len(facts_ejemplo)}")
facts_ejemplo.head()

Total de 'hechos' (facts) en este archivo: 6648


,nombre,valor_texto,contextID,es_instante,es_duracion,inicio,fin,num_dimensiones,es_total_efectivo,valor
0,co-sfc-core:ActividadPrincipal,"""Las principales actividades de ISA y sus subs...",TrimestreAcumuladoActual,False,True,2023-01-01,2024-01-01,0,True,NaN
1,co-sfc-core:ActivosFinancierosCorrientesAlCost...,4071927866,CierreTrimestreActual,True,False,NaT,2024-01-01,0,True,4.071928e+09
2,co-sfc-core:ActivosFinancierosCorrientesAlCost...,5369349875,SaldoActualInicio,True,False,NaT,2023-01-01,0,True,5.369350e+09
3,co-sfc-core:ActivosVinculadosNegociosConjuntos,31553430738,CierreTrimestreActual,True,False,NaT,2024-01-01,0,True,3.155343e+10
4,co-sfc-core:ActivosVinculadosNegociosConjuntos,33748292506,SaldoActualInicio,True,False,NaT,2023-01-01,0,True,3.374829e+10


## 1.3. Los conceptos de la taxonomía que usamos

| Nuestro campo | Concepto(s) XBRL |
|---|---|
| Activo total | `ifrs:Assets` |
| Activo corriente | `ifrs:CurrentAssets` |
| Pasivo total | `ifrs:Liabilities` |
| Pasivo corriente / no corriente | `ifrs:CurrentLiabilities` / `ifrs:NoncurrentLiabilities` |
| Patrimonio | `ifrs:Equity` |
| Ganancias retenidas | `ifrs:RetainedEarnings` |
| Efectivo | `ifrs:CashAndCashEquivalents` |
| Inventarios | `ifrs:Inventories` |
| Deuda corto / largo plazo | `co-sfc-core:ObligacionesFinancierasCorrientes` / `...NoCorrientes` (extensión local de la Superfinanciera), con `ifrs:ShorttermBorrowings` / `ifrs:LongtermBorrowings` como respaldo |
| Ingresos | `ifrs:Revenue` |
| Utilidad neta | `ifrs:ProfitLossAttributableToOwnersOfParent` **(no** `ifrs:ProfitLoss`, que es la utilidad total del grupo antes de repartir con el interés minoritario)|
| Gastos financieros | `ifrs:FinanceCosts` |
| Impuesto de renta | `ifrs:IncomeTaxExpenseContinuingOperations` |
| Depreciación y amortización | `ifrs:DepreciationAndAmortisationExpense`, con respaldo sumando `DepreciationExpense` + `AmortisationExpense` por separado cuando la empresa no reporta la versión combinada |

**Nota sobre ISA e "ingresos sin construcción":** la versión del proyecto basada en PDF usaba el
ingreso "sin construcción" que ISA reporta en su tabla ejecutiva para inversionistas (excluye el
ingreso contable de construir el activo de la concesión bajo NIIF/IFRIC 12, que no tiene margen
real). Se revisó a fondo si esa desagregación existe en el XBRL -tanto como concepto propio, como
desglose dimensional dentro de `ifrs:Revenue`- y no aparece en ningún lado: es un dato que ISA solo
presenta en su reporte ejecutivo para inversionistas, no en los estados financieros formales que se
etiquetan en XBRL. Por eso, para ISA usamos aquí `ifrs:Revenue` (ingreso total, con construcción)
sin ese ajuste -es una diferencia metodológica menor y deliberada frente a la versión con PDF-.

In [4]:
facts_ejemplo[facts_ejemplo["nombre"].isin(["ifrs:Assets","ifrs:Equity","ifrs:Revenue"])
             ][["nombre","valor","es_instante","fin","num_dimensiones"]].head(10)

,nombre,valor,es_instante,fin,num_dimensiones
678,ifrs:Assets,7.121718e+10,True,2024-01-01,0
679,ifrs:Assets,7.873385e+10,True,2023-01-01,0
2143,ifrs:Equity,2.636639e+10,True,2024-01-01,0
2144,ifrs:Equity,0.000000e+00,True,2024-01-01,2
2145,ifrs:Equity,0.000000e+00,True,2023-01-01,2
2146,ifrs:Equity,0.000000e+00,True,2024-01-01,2
2147,ifrs:Equity,0.000000e+00,True,2023-01-01,2
2148,ifrs:Equity,0.000000e+00,True,2024-01-01,2
2149,ifrs:Equity,0.000000e+00,True,2023-01-01,2
2150,ifrs:Equity,0.000000e+00,True,2024-01-01,2


## 1.4. La escala: un elemento XBRL dedicado, no una frase suelta

En el PDF, la unidad ("expresado en miles/millones de pesos") es una frase de texto libre que hay
que buscar y que a veces se mezcla con menciones de otras cifras en millones dentro de la misma
página. En XBRL existe un elemento estructurado y dedicado para esto,
`RoundingUsedInFinancialStatements`, que cada emisor declara explícitamente:

In [5]:
for empresa in ["ISA","CELSIA","EPM","ISAGEN","ENEL"]:
    escala = x.detectar_escala(f"../data_xbrl/{empresa}/2023/Q2 2023.xbrl")
    print(f"{empresa:10} -> factor para llevar a millones: {escala}")

ISA        -> factor para llevar a millones: 0.001
CELSIA     -> factor para llevar a millones: 0.001
EPM        -> factor para llevar a millones: 0.001
ISAGEN     -> factor para llevar a millones: 1e-06
ENEL       -> factor para llevar a millones: 0.001


**Ojo con ISAGEN**: declara *"Pesos"* (sin ningún tipo de escalamiento), así que su factor es
0.000001 -si se tratara como si ya viniera en millones, los valores quedarían un millón de veces más
grandes de lo real-. Confirmamos este detalle cruzando manualmente el activo total de ISAGEN contra
su PDF antes de generalizarlo (ver conversación del proyecto).

## 1.5. Extracción completa de los 146 archivos

Los 5 emisores cubren de 2019 a 2026 (CELSIA no tiene 2019 disponible en XBRL, y por eso tiene 26
archivos en vez de 30). Para ISAGEN, del cierre de 2023 en adelante los archivos son Consolidados;
antes de esa fecha son Separados -se validó por separado que, para ISAGEN específicamente, la
diferencia entre Separado y Consolidado es mínima (bajo 1% en la mayoría de las partidas), porque no
tiene participaciones societarias materiales, así que mezclar ambas fuentes en la misma serie no
introduce una distorsión real-.

In [6]:
df_crudo = build_dataset_xbrl.main()

Procesando: CELSIA/2020/CIERRE 2020.xbrl


Procesando: CELSIA/2020/Q1 2020.xbrl


Procesando: CELSIA/2020/Q2 2020.xbrl


Procesando: CELSIA/2020/Q3 2020.xbrl


Procesando: CELSIA/2021/CIERRE 2021.xbrl


...

Procesando: ISAGEN/2025/Q2 2025.xbrl


Procesando: ISAGEN/2025/Q3 2025.xbrl


Procesando: ISAGEN/2026/Q1 2026.xbrl


Procesando: ISAGEN/2026/Q2 2026.xbrl



Listo: 146 filas guardadas en /home/claude/repo_xbrl/src/../data_xbrl_out/raw_quarterly_xbrl.csv


In [7]:
df_crudo.head()

,empresa,año,trimestre,archivo,escala_detectada,efectivo,activo_corriente,activo_total,pasivo_corriente,pasivo_no_corriente,pasivo_total,patrimonio_total,ganancias_retenidas,inventarios,deuda_cp,deuda_lp,ingresos,utilidad_neta,utilidad_operativa,gastos_financieros,impuesto_renta,depreciacion_amortizacion
0,CELSIA,2020,1,Q1 2020.xbrl,0.001,502916.479,2031292.006,1.225894e+07,2281054.158,4315362.613,6596416.771,5662526.203,323020.110,221393.090,836886.912,3653404.723,1184559.910,65028.208,226528.173,-95978.231,-66916.909,12126.532
1,CELSIA,2020,2,Q2 2020.xbrl,0.001,576094.321,2052267.789,1.214731e+07,2067633.516,4422371.541,6490005.057,5657309.932,323020.110,229634.211,819121.849,3761977.776,1819462.234,138120.576,458782.014,-213920.462,-117404.855,25882.089
2,CELSIA,2020,3,Q3 2020.xbrl,0.001,372786.178,1545443.512,1.213915e+07,1818980.278,4521253.793,6340234.071,5798917.811,323020.110,265853.190,713308.386,3854232.743,2638470.706,193792.100,625657.047,-251881.639,-144659.292,40978.649
3,CELSIA,2020,4,CIERRE 2020.xbrl,0.001,399547.205,1426594.051,1.181066e+07,2055711.187,4014292.754,6070003.941,5740655.566,323020.110,167135.929,483336.146,3382519.223,3643877.370,249319.973,877364.881,-354770.516,-204354.994,55553.718
4,CELSIA,2021,1,Q1 2021.xbrl,0.001,283111.862,1516662.477,1.227323e+07,2319218.414,4347552.752,6666771.166,5606460.057,289816.424,178356.972,498103.938,3720381.124,995146.453,83504.070,221974.073,-71337.951,-53947.938,12641.397


### Qué tan completos quedaron los datos

In [8]:
campos_clave = ["activo_total","pasivo_total","patrimonio_total","efectivo","deuda_cp","deuda_lp",
                "ingresos","utilidad_neta","gastos_financieros","depreciacion_amortizacion",
                "activo_corriente","pasivo_corriente","ganancias_retenidas"]

resumen = []
for empresa in df_crudo["empresa"].unique():
    sub = df_crudo[df_crudo.empresa == empresa]
    total_celdas = len(sub) * len(campos_clave)
    nulos = sub[campos_clave].isna().sum().sum()
    resumen.append({"empresa": empresa, "trimestres": len(sub),
                     "% completitud": round(100 * (1 - nulos / total_celdas), 1)})

pd.DataFrame(resumen)

,empresa,trimestres,% completitud
0,CELSIA,26,100.0
1,ENEL,30,91.3
2,EPM,30,94.1
3,ISA,30,100.0
4,ISAGEN,30,100.0


### Limitación de la fuente, confirmada -no es un error nuestro de extracción

En 4 de los 146 archivos, el contenido interno no corresponde al período que indica el nombre del
archivo tal como se descargó del RNVE:

| Archivo | Debía cerrar en | Lo que realmente contiene |
|---|---|---|
| ENEL/2019/Q3 2019.xbrl | 30-sep-2019 | 30-jun-2019 (un trimestre antes) |
| EPM/2019/Q2 2019.xbrl | 30-jun-2019 | 30-jun-2018 (un año antes) |
| EPM/2019/Q3 2019.xbrl | 30-sep-2019 | 30-sep-2018 (un año antes) |
| EPM/2020/Q1 2020.xbrl | 31-mar-2020 | 31-dic-2019 (el cierre anterior) |

Se confirmó con un hash MD5 idéntico entre la descarga original y una redescarga posterior del
archivo de ENEL que esto **no es un error de descarga**: es lo que la Superfinanciera tiene
registrado bajo ese nombre en su histórico -muy probablemente un error de transmisión de la propia
empresa en 2019-2020 que nunca se corrigió-. Estos 4 trimestres quedan con los campos del estado de
resultados y del balance vacíos, en vez de usar un dato que sabemos que corresponde a otro período.

In [9]:
faltantes = ["ENEL/2019/Q3 2019.xbrl", "EPM/2019/Q2 2019.xbrl", "EPM/2019/Q3 2019.xbrl", "EPM/2020/Q1 2020.xbrl"]
df_crudo[df_crudo["archivo"].isin([f.split("/")[-1] for f in faltantes])][
    ["empresa","año","trimestre","archivo","activo_total","ingresos"]]

,empresa,año,trimestre,archivo,activo_total,ingresos
0,CELSIA,2020,1,Q1 2020.xbrl,1.225894e+07,1.184560e+06
27,ENEL,2019,2,Q2 2019.xbrl,8.812664e+06,1.932017e+06
28,ENEL,2019,3,Q3 2019.xbrl,NaN,NaN
30,ENEL,2020,1,Q1 2020.xbrl,9.268463e+06,1.067508e+06
57,EPM,2019,2,Q2 2019.xbrl,NaN,NaN
58,EPM,2019,3,Q3 2019.xbrl,NaN,1.329104e+07
60,EPM,2020,1,Q1 2020.xbrl,5.717288e+07,4.746387e+06
87,ISA,2019,2,Q2 2019.xbrl,4.726566e+07,3.917489e+06
88,ISA,2019,3,Q3 2019.xbrl,4.824042e+07,5.888838e+06
90,ISA,2020,1,Q1 2020.xbrl,5.134027e+07,2.069573e+06


## 1.6. De acumulado a trimestre aislado, LTM, y EBITDA reconstruido

Igual que en la versión con PDF: el estado de resultados viene acumulado del año, así que primero
se "desacumula" a trimestre aislado, y de ahí se arma la ventana LTM (últimos 12 meses).

$$\text{Trimestre aislado}_n = \text{Acumulado}_n - \text{Acumulado}_{n-1}$$

**Diferencia frente al PDF:** ninguna de las 5 empresas tiene una etiqueta XBRL de "EBITDA" como
tal -no es un concepto de la taxonomía IFRS-, así que aquí el EBITDA se reconstruye con el método
indirecto para las 5 por igual, en vez de usar el EBITDA reportado como texto para CELSIA e ISA y el
método indirecto solo para las otras 3 (que era la mezcla que tocaba hacer con los PDF):

$$\text{EBITDA} = \text{Utilidad neta} + \text{Impuesto de renta} + \text{Gastos financieros netos} + \text{D\&A}$$

Esto es, de hecho, más consistente metodológicamente entre las 5 empresas que lo que teníamos antes.

In [10]:
df_ratios = compute_ratios_xbrl.main()

Listo: 146 filas guardadas en /home/claude/repo_xbrl/src/../data_xbrl_out/ratios_xbrl.csv

Trimestres marcados para revisión manual: 15 de 146
empresa  año  trimestre          archivo revisar_manualmente
 CELSIA 2022          4 CIERRE 2022.xbrl  utilidad_neta_trim
 CELSIA 2024          1     Q1 2024.xbrl  utilidad_neta_trim
   ENEL 2022          3     Q3 2022.xbrl       ingresos_trim
   ENEL 2024          4 CIERRE 2024.xbrl  utilidad_neta_trim
   ENEL 2025          4 CIERRE 2025.xbrl  utilidad_neta_trim
    EPM 2021          4 CIERRE 2021.xbrl  utilidad_neta_trim
    EPM 2023          4 CIERRE 2023.xbrl  utilidad_neta_trim
    EPM 2024          4 CIERRE 2024.xbrl  utilidad_neta_trim
    EPM 2025          3     Q3 2025.xbrl  utilidad_neta_trim
    ISA 2021          3     Q3 2021.xbrl  utilidad_neta_trim
    ISA 2022          3     Q3 2022.xbrl  utilidad_neta_trim
 ISAGEN 2020          2     Q2 2020.xbrl  utilidad_neta_trim
 ISAGEN 2021          2     Q2 2021.xbrl  utilidad_neta_trim
 IS

In [11]:
df_ratios.head()

,empresa,año,trimestre,archivo,escala_detectada,efectivo,activo_corriente,activo_total,pasivo_corriente,pasivo_no_corriente,pasivo_total,patrimonio_total,ganancias_retenidas,inventarios,deuda_cp,...,cobertura_intereses,ffo_intereses,ffo_deuda,razon_corriente,prueba_acida,caja_activos,caja_deuda_cp,margen_ebitda,margen_neto,zscore_x1,zscore_x2,zscore_x3,zscore_x4,zscore,zscore_zona
0,CELSIA,2020,1,Q1 2020.xbrl,0.001,502916.479,2031292.006,1.225894e+07,2281054.158,4315362.613,6596416.771,5662526.203,323020.110,221393.090,836886.912,...,NaN,NaN,NaN,0.890506,0.793448,0.041024,0.600937,NaN,NaN,-0.020374,0.026350,NaN,0.858425,NaN,NaN
1,CELSIA,2020,2,Q2 2020.xbrl,0.001,576094.321,2052267.789,1.214731e+07,2067633.516,4422371.541,6490005.057,5657309.932,323020.110,229634.211,819121.849,...,NaN,NaN,NaN,0.992568,0.881507,0.047426,0.703307,NaN,NaN,-0.001265,0.026592,NaN,0.871696,NaN,NaN
2,CELSIA,2020,3,Q3 2020.xbrl,0.001,372786.178,1545443.512,1.213915e+07,1818980.278,4521253.793,6340234.071,5798917.811,323020.110,265853.190,713308.386,...,NaN,NaN,NaN,0.849621,0.703466,0.030709,0.522616,NaN,NaN,-0.022533,0.026610,NaN,0.914622,NaN,NaN
3,CELSIA,2020,4,CIERRE 2020.xbrl,0.001,399547.205,1426594.051,1.181066e+07,2055711.187,4014292.754,6070003.941,5740655.566,323020.110,167135.929,483336.146,...,2.435375,0.859355,0.078863,0.693966,0.612663,0.033829,0.826645,0.237110,0.068422,-0.053267,0.027350,0.074286,0.945742,4.481959,segura
4,CELSIA,2021,1,Q1 2021.xbrl,0.001,283111.862,1516662.477,1.227323e+07,2319218.414,4347552.752,6666771.166,5606460.057,289816.424,178356.972,498103.938,...,2.560749,0.981020,0.076773,0.653954,0.577050,0.023067,0.568379,0.244721,0.077522,-0.065391,0.023614,0.071115,0.840956,4.258914,segura


### Trimestres marcados para revisión manual

Igual que antes, se compara cada trimestre contra el mismo trimestre del año anterior (no contra el
trimestre inmediatamente anterior, por la estacionalidad hidrológica real del sector). Se verificó
que ningún trimestre restante produce un valor de ratio fuera de rango razonable en toda la serie
histórica.

In [12]:
marcados = df_ratios[df_ratios["revisar_manualmente"] != ""]
print(f"Trimestres marcados: {len(marcados)} de {len(df_ratios)} ({100*len(marcados)/len(df_ratios):.0f}%)\n")
marcados[["empresa", "año", "trimestre", "archivo", "revisar_manualmente"]]

Trimestres marcados: 15 de 146 (10%)



,empresa,año,trimestre,archivo,revisar_manualmente
11,CELSIA,2022,4,CIERRE 2022.xbrl,utilidad_neta_trim
16,CELSIA,2024,1,Q1 2024.xbrl,utilidad_neta_trim
40,ENEL,2022,3,Q3 2022.xbrl,ingresos_trim
49,ENEL,2024,4,CIERRE 2024.xbrl,utilidad_neta_trim
53,ENEL,2025,4,CIERRE 2025.xbrl,utilidad_neta_trim
67,EPM,2021,4,CIERRE 2021.xbrl,utilidad_neta_trim
75,EPM,2023,4,CIERRE 2023.xbrl,utilidad_neta_trim
79,EPM,2024,4,CIERRE 2024.xbrl,utilidad_neta_trim
82,EPM,2025,3,Q3 2025.xbrl,utilidad_neta_trim
96,ISA,2021,3,Q3 2021.xbrl,utilidad_neta_trim


## Conclusión de esta sección

Quedan guardados `data_xbrl_out/raw_quarterly_xbrl.csv` y `data_xbrl_out/ratios_xbrl.csv`. El
notebook `02_analisis_y_modelos.ipynb` parte directamente de `ratios_xbrl.csv` para los gráficos de
evolución, el Z''-Score y el modelo de Merton/KMV.